In [14]:
# Importación de librerías requeridas
import pandas as pd
import numpy as np
import joblib

# Módulos para preparación, modelado y evaluación
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [15]:
# 1. Cargar el dataset
df = pd.read_csv('dataset_precio_gasolina.csv')

# Visualizar las primeras filas del conjunto de datos
print("--- PRIMERAS FILAS DEL DATASET ---")
display(df.head())

# 2. Tratamiento de valores faltantes (limpieza de datos)
df = df.dropna()

# 3. Definición de variables según el diagrama:
# y = Target / Variable dependiente
# X = Features / Variables independientes
y = df['Precio_Gasolina_USD_Galon']
X = df.drop(columns=['Fecha', 'Precio_Gasolina_USD_Galon'])

print("\nForma de X (Matriz de características):", X.shape)
print("Forma de y (Vector objetivo):", y.shape)

--- PRIMERAS FILAS DEL DATASET ---


,Fecha,Precio_Gasolina_USD_Galon,Precio_Petroleo_USD_Barril,Tipo_Cambio_USD_COP,Demanda_miles_barriles_dia,Produccion_OPEP_millones_dia,Inflacion_Porcentaje,PIB_Porcentaje,Eventos_Geopoliticos,Impuestos_Combustibles_Porcentaje
0,2018-01-01,8.17,73.94,3683.88,9673.0,27.45,3.84,3.54,1,18
1,2018-02-01,8.51,62.51,4404.09,9819.0,28.53,2.75,5.57,0,22
2,2018-03-01,8.17,76.66,2521.11,9254.0,28.03,3.25,2.28,0,22
3,2018-04-01,9.09,92.41,4069.86,9230.0,29.16,2.91,3.30,0,22
4,2018-05-01,8.10,60.79,3739.17,9746.0,33.63,2.79,3.88,0,18



Forma de X (Matriz de características): (72, 8)
Forma de y (Vector objetivo): (72,)


In [16]:
# Crear y ajustar el escalador estándar
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Exportar el escalador para que la App de Streamlit normalice los datos ingresados por el usuario
joblib.dump(scaler, 'scaler_gasolina.joblib')
print("Escalador guardado correctamente como 'scaler_gasolina.joblib'")

Escalador guardado correctamente como 'scaler_gasolina.joblib'


In [45]:
# Partición 70% entrenamiento y 30% evaluación (indicado en el tablero)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.35, random_state=14
)

# Instanciar el algoritmo candidatos (Regresión Lineal)
modelo_rl = LinearRegression()

# Entrenar el modelo
modelo_rl.fit(X_train, y_train)

print("Entrenamiento completado exitosamente con 70% de datos de entrenamiento.")

Entrenamiento completado exitosamente con 70% de datos de entrenamiento.


In [46]:
# Realizar predicciones sobre el conjunto de prueba
y_pred = modelo_rl.predict(X_test)

# Cálculo de las métricas solicitadas
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("========================================")
print("       MÉTRICAS DE EVALUACIÓN          ")
print("========================================")
print(f"R² (Coeficiente de Determinación): {r2:.4f}")
print(f"MAE (Error Absoluto Medio):       {mae:.4f}")
print(f"MSE (Error Cuadrático Medio):      {mse:.4f}")
print("========================================")

       MÉTRICAS DE EVALUACIÓN          
R² (Coeficiente de Determinación): 0.9421
MAE (Error Absoluto Medio):       0.1629
MSE (Error Cuadrático Medio):      0.0451


In [47]:
# Cálculo del error estándar a partir de los residuos en entrenamiento
residuos = y_train - modelo_rl.predict(X_train)
std_residuos = np.std(residuos)

# Margen de error al 95% de confianza (Z = 1.96)
margen_error_95 = 1.96 * std_residuos

# Guardar el modelo entrenado usando Joblib (.joblib / .pkl)
joblib.dump(modelo_rl, 'modelo_gasolina.joblib')

# Guardar información requerida por la App
info_despliegue = {
    'margen_error_95': margen_error_95,
    'columnas': list(X.columns)
}
joblib.dump(info_despliegue, 'info_despliegue.joblib')

print(f"Margen de error (95% confianza): ± {margen_error_95:.4f} USD/galón")
print("Modelo guardado como 'modelo_gasolina.joblib'.")

Margen de error (95% confianza): ± 0.4973 USD/galón
Modelo guardado como 'modelo_gasolina.joblib'.
